# Practice 42 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — planets: string ops

Load `sns.load_dataset('planets')`.

The `method` column has values like `'Radial Velocity'`, `'Transit'`, `'Imaging'` etc.

1. Use `.str.contains()` to flag planets discovered by a method that includes the word `'Velocity'`. How many?
2. Use `.str.extract()` to pull the **first word** of each `method` into a new column `method_short`.
3. Use `.str.contains()` to find methods with more than one word (hint: contains a space). How many unique multi-word methods are there? Use `.unique()` on the filtered column.
4. Add `decade` from `year`: extract the decade as a string (e.g. 2005 → `'2000s'`). Hint: extract the first 3 digits with `.str.extract()` on the string-cast year, then append `'0s'`.
5. Which `decade` has the most planet discoveries? No groupby — use `.value_counts()`.

In [23]:
# Your code here

planets = sns.load_dataset('planets')

vp = planets[planets['method'].str.contains('Velocity')]
vpnum = vp.shape[0]

planets['method_short'] = planets['method'].str.extract(r'(\w+)')
mw = planets[planets['method'].str.contains(' ')]
mw['method'].unique()

planets['decade'] = planets['year'].astype(str).str.extract(r'(\d{3})')+'0s'

planets['decade'].value_counts().idxmax()

'2010s'

---
## Level 2 — employee data: extract + classify

You have an employee dataset with messy string columns. No `.apply()`.

1. Use `.str.extract()` to pull the numeric part from `employee_id` into `id_num` (as integer).
2. Use `.str.extract()` to pull the **department code** (the letters before the dash in `employee_id`) into `dept_code`.
3. Use `.str.contains()` to flag employees whose `email` is a gmail address.
4. Use `.str.extract()` to pull the **domain** from each email (everything after `@`).
5. Use `np.select()` to add `seniority`: `'senior'` (years_exp >= 8), `'mid'` (years_exp >= 3), `'junior'` otherwise. Then convert `seniority` and `dept_code` to `category`.
6. Build a dict comprehension `{dept_code: mean_salary}` from the groupby result.

In [51]:
employees = pd.DataFrame({
    'employee_id': ['ENG-001', 'MKT-002', 'ENG-003', 'HR-004', 'MKT-005',
                    'HR-006', 'ENG-007', 'FIN-008', 'MKT-009', 'FIN-010'],
    'name':        ['Alice', 'Bob', 'Carol', 'David', 'Eva',
                    'Frank', 'Grace', 'Hiro', 'Ivy', 'Jack'],
    'email':       ['alice@gmail.com', 'bob@company.com', 'carol@gmail.com',
                    'david@company.com', 'eva@outlook.com', 'frank@gmail.com',
                    'grace@company.com', 'hiro@outlook.com', 'ivy@gmail.com', 'jack@company.com'],
    'years_exp':   [5, 2, 11, 7, 1, 9, 3, 14, 2, 8],
    'salary':      [85000, 52000, 120000, 78000, 45000, 105000, 62000, 135000, 48000, 98000]
})

# Your code here
employees['id_num'] = employees['employee_id'].str.extract(r'(\d+)').astype(int)
employees['dept_code'] = employees['employee_id'].str.extract(r'^([A-Z]+)')
gf = employees['email'].str.contains('gmail.com')
domain = employees['email'].str.extract(r'@(.+)') 

conds = [employees['years_exp']>=8, employees['years_exp']>=3]
chos = ['senior', 'mid']
employees['seniority'] =np.select(conds, chos, default='junior')
employees[['dept_code','seniority']] = employees[['dept_code','seniority']].astype('category')

g = employees.groupby("dept_code",observed=True)['salary'].mean()
{dept: ms for dept, ms in zip(g.index, g)}

{'ENG': 89000.0, 'FIN': 116500.0, 'HR': 91500.0, 'MKT': 48333.333333333336}

---
## Level 3 — pipeline on messy log data

You have a server log dataset below. Write a `.pipe()` chain — no `.apply()`.

- **`parse(df)`** — use `.str.extract()` to pull:
  - `status_code` (the 3-digit number in `log_entry`) as integer
  - `endpoint` (the path after `GET` or `POST`, e.g. `/api/users`) — pattern: `r'(?:GET|POST) (\S+)'`
  - `method` (GET or POST) — pattern: `r'(GET|POST)'`
- **`classify(df)`** — add `status_class` with `np.select()`: `'success'` (status_code < 300), `'redirect'` (status_code < 400), `'client_error'` (status_code < 500), `'server_error'` otherwise. Add `is_api`: True where `endpoint` contains `'/api/'`.

After the chain:
- What fraction of API requests (`is_api == True`) resulted in an error (status_code >= 400)? Use `.loc` and `np.mean()`.
- Use `.value_counts()` to see the most common endpoints.
- Use a set comprehension to find all unique `status_class` values for POST requests.

In [52]:
logs = pd.DataFrame({
    'log_entry': [
        'GET /api/users 200',
        'POST /api/login 201',
        'GET /home 200',
        'GET /api/data 404',
        'POST /api/users 500',
        'GET /about 200',
        'GET /api/stats 200',
        'POST /api/login 401',
        'GET /home 301',
        'POST /api/data 503',
        'GET /api/users 200',
        'GET /contact 200',
    ]
})

# Your code here

def parse(df):
    df['status_code'] = df['log_entry'].str.extract(r'(\d{3}$)').astype(int)
    df['endpoint'] = df['log_entry'].str.extract(r'(?:GET|POST) (\S+)')
    df['method'] = df['log_entry'].str.extract(r'(GET|POST)')
    return df 

def classify(df):
    conds = [df['status_code']<300, df['status_code']<400, df['status_code']<500]
    chos = ['success','redirect','client_error']
    df['status_class'] = np.select(conds, chos, default = "server_error")
    df['is_api'] = df['endpoint'].str.contains('/api/')
    return df 


res = logs.copy().pipe(parse).pipe(classify)

np.mean(res.loc[res['is_api'],'status_code']>=400)

res['endpoint'].value_counts().idxmax()

{sc for sc, m in zip (res['status_class'],res['method']) if m=='POST'}

{'client_error', 'server_error', 'success'}